[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/matibilkis/RL-UBA-2026/blob/main/clase_8/comparacion_cartpole.ipynb)

# CartPole: Q-learning, REINFORCE sin baseline y REINFORCE con baseline

Clase 8. Tres algoritmos sobre el mismo entorno: Q-learning discretizando el estado continuo, y REINFORCE (con y sin baseline) con una política lineal softmax hecha a mano, sin redes (la Clase 7 tal cual, más una sola línea para el baseline).

Una sola semilla por algoritmo: demo didáctica para ver la forma de las curvas, no un benchmark.

In [ ]:
%pip install -q gymnasium  # no viene preinstalado en Colab; numpy/matplotlib sí

In [ ]:
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

GAMMA = 0.99
VENTANA = 100   # episodios de la media móvil -- la misma ventana para las tres curvas


def media_movil(x, w):
    """Promedio de cada ventana de w valores consecutivos, sin loop: suma acumulada y resto la
    ventana que queda atrás -- el mismo resultado que convolucionar con un kernel de unos, pero
    se lee directo."""
    c = np.cumsum(np.insert(x, 0, 0.0))
    return (c[w:] - c[:-w]) / w


def suavizar(puntajes, pasos, w=VENTANA):
    """Media móvil de w episodios + los steps que le corresponden a cada punto -- las dos
    salidas quedan siempre del mismo largo (se recortan juntas los primeros w-1 episodios)."""
    return pasos[w - 1:], media_movil(puntajes, w)


def resuelto_en(pasos_suave, media_suave, umbral=475):
    """Primer step de la curva ya suavizada donde el promedio llega al umbral estándar de Gymnasium."""
    i = np.argmax(media_suave >= umbral) if (media_suave >= umbral).any() else None
    return None if i is None else int(pasos_suave[i])

## 1. Q-learning tabular, discretizando el estado continuo de CartPole

El estado es (posición, velocidad, ángulo, velocidad angular). Lo partimos en una grilla de cajas: casi todo el peso lo ponemos en el ángulo (lo que más importa para no caerse), poco en posición/velocidad.

In [ ]:
BINS = (1, 1, 6, 12)
LIMS = [(-2.4, 2.4), (-3.0, 3.0), (-0.21, 0.21), (-3.0, 3.0)]


def discretizar(obs):
    """Un estado continuo -> una tupla de índices de caja, uno por dimensión."""
    return tuple(min(b - 1, max(0, int((o - lo) / (hi - lo) * b))) for o, (lo, hi), b in zip(obs, LIMS, BINS))


def q_learning(episodios, alpha=0.05, eps0=1.0, eps_min=0.02, eps_decay=0.999, semilla=0, sarsa=False):
    """Q-learning (off-policy) o Sarsa (on-policy) sobre la grilla de BINS. La única diferencia está
    marcada abajo: Q-learning usa el máximo sobre acciones del estado siguiente; Sarsa usa la acción
    que ε-greedy realmente va a tomar ahí."""
    rng = np.random.default_rng(semilla)
    env = gym.make("CartPole-v1")
    Q = np.zeros(BINS + (2,))
    eps = eps0
    puntajes = np.zeros(episodios)
    pasos = np.zeros(episodios, dtype=int)
    total = 0
    for e in range(episodios):
        s = discretizar(env.reset(seed=int(rng.integers(1 << 31)))[0])
        a = rng.integers(2) if rng.random() < eps else int(Q[s].argmax())
        G = 0.0
        while True:
            o2, r, term, trunc, _ = env.step(int(a))
            total += 1
            s2 = discretizar(o2)
            a2 = rng.integers(2) if rng.random() < eps else int(Q[s2].argmax())     # ε-greedy, se usa para actuar siempre
            objetivo = Q[s2][a2] if sarsa else Q[s2].max()                          # <- la única línea que cambia: Sarsa (on-policy) vs Q-learning (off-policy)
            Q[s][a] += alpha * (r + GAMMA * objetivo * (not term) - Q[s][a])
            s, a, G = s2, a2, G + r
            if term or trunc:
                break
        puntajes[e] = G
        pasos[e] = total
        eps = max(eps_min, eps * eps_decay)
    return puntajes, pasos, Q

In [ ]:
EP_Q = 8000
puntajes_q, pasos_q, Q_tabla = q_learning(EP_Q)
print(f"Q-learning: {EP_Q} episodios, {pasos_q[-1]} steps. Puntaje medio últimos {VENTANA} episodios: {puntajes_q[-VENTANA:].mean():.1f}")

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(media_movil(puntajes_q, VENTANA), color="tab:red", linewidth=2, alpha=0.8)
ax.set(xlabel="episodio", ylabel="puntaje (máx. 500)", title="Q-learning discretizado")
fig.tight_layout()
plt.show()

## 2. REINFORCE sin baseline: política lineal softmax, a mano

Misma política de la Clase 7: features = las 4 observaciones + bias, pesos $\theta$ de 5×2, una softmax a mano sobre los dos logits.

In [ ]:
def softmax(z):
    """z es pesos @ features (2 acciones). El z.max() es solo estabilidad numérica."""
    e = np.exp(z - z.max())
    return e / e.sum()


def reinforce(episodios, alpha=3e-3, baseline=False, semilla=0):
    """Una actualización por episodio. Con baseline=False es la Clase 7 tal cual. Con baseline=True,
    se le resta al retorno un promedio móvil (EMA) de los retornos pasados -- una sola línea de más."""
    rng = np.random.default_rng(semilla)
    env = gym.make("CartPole-v1")
    theta = np.zeros((5, 2))                              # 4 observaciones + 1 (bias), 2 acciones
    puntajes = np.zeros(episodios)
    pasos = np.zeros(episodios, dtype=int)
    total = 0
    b = 0.0
    for e in range(episodios):
        o, _ = env.reset(seed=int(rng.integers(1 << 31)))
        X, A, R = [], [], []
        while True:
            x = np.append(o, 1.0)
            a = rng.choice(2, p=softmax(x @ theta))       # acción discreta: 0 = empujar a izquierda, 1 = a derecha
            X.append(x)
            A.append(a)
            o, r, term, trunc, _ = env.step(int(a))
            R.append(r)
            total += 1
            if term or trunc:
                break
        X = np.array(X)
        G = np.zeros(len(R))
        g = 0.0
        for t in reversed(range(len(R))):                                          # retorno desde cada paso, hacia atrás
            g = R[t] + GAMMA * g
            G[t] = g
        if baseline:
            b = 0.9 * b + 0.1 * G[0]                                               # EMA del retorno del episodio anterior
        ventaja = G - b                                                            # con baseline=False, b se queda en 0: es sacar la cuenta de más
        pesos = GAMMA ** np.arange(len(R)) * ventaja
        P = np.array([softmax(x @ theta) for x in X])
        residual = np.eye(2)[A] - P                                                # score de la softmax: 1{a} - π(a|s)
        theta += alpha * X.T @ (pesos[:, None] * residual)
        puntajes[e] = sum(R)
        pasos[e] = total
    return puntajes, pasos

In [ ]:
EP_PG = 8000
puntajes_sb, pasos_sb = reinforce(EP_PG, baseline=False)
print(f"REINFORCE sin baseline: {EP_PG} episodios, {pasos_sb[-1]} steps. Puntaje medio últimos {VENTANA}: {puntajes_sb[-VENTANA:].mean():.1f}")

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(media_movil(puntajes_sb, VENTANA), color="tab:blue", linewidth=2, alpha=0.8)
ax.set(xlabel="episodio", ylabel="puntaje (máx. 500)", title="REINFORCE sin baseline")
fig.tight_layout()
plt.show()

## 3. REINFORCE con baseline: la misma función, un solo argumento distinto

In [ ]:
puntajes_cb, pasos_cb = reinforce(EP_PG, baseline=True)
print(f"REINFORCE con baseline: {EP_PG} episodios, {pasos_cb[-1]} steps. Puntaje medio últimos {VENTANA}: {puntajes_cb[-VENTANA:].mean():.1f}")

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(media_movil(puntajes_cb, VENTANA), color="black", linewidth=2, alpha=0.8)
ax.set(xlabel="episodio", ylabel="puntaje (máx. 500)", title="REINFORCE con baseline")
fig.tight_layout()
plt.show()

## 4. Comparación

Las tres curvas en función de los steps de entorno consumidos (no de episodios: cada algoritmo necesita una cantidad de episodios muy distinta, pero el "costo" real son los steps).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
corridas = (
    (pasos_q, puntajes_q, "tab:red", "Q-learning"),
    (pasos_sb, puntajes_sb, "tab:blue", "REINFORCE sin baseline"),
    (pasos_cb, puntajes_cb, "black", "REINFORCE con baseline"),
)
for pasos, puntajes, color, nombre in corridas:
    pasos_suave, media_suave = suavizar(puntajes, pasos)
    ax.plot(pasos_suave, media_suave, label=nombre, color=color, linewidth=2, alpha=0.8)
ax.set(xlabel="steps de entorno (acumulados)", ylabel=f"puntaje (media móvil de {VENTANA} episodios)", title="CartPole: los tres algoritmos")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
# OJO: 'steps totales' de Q-learning incluye entrenar MUCHO después de resolverlo (episodios ya
# resueltos duran 500 pasos cada uno); 'resuelto en' es la comparación justa entre los tres.
print(f"Resumen -- 'resuelto en' = primer step donde la media móvil de {VENTANA} episodios llega a 475")
for pasos, puntajes, _, nombre in corridas:
    pasos_suave, media_suave = suavizar(puntajes, pasos)
    step_r = resuelto_en(pasos_suave, media_suave)
    resuelto = f"resuelto en step {step_r}" if step_r else "no llegó a 475 de promedio"
    print(f"  {nombre:24s}  puntaje final {puntajes[-VENTANA:].mean():6.1f}   steps totales {pasos[-1]:>8d}   {resuelto}")

## 5. Opcional -- Sarsa

UNA línea de diferencia con Q-learning (ver el comentario marcado arriba, en `q_learning`). No agrega un eje nuevo a esta comparación (sigue siendo "basado en valor", como Q-learning) y la Clase 6 ya vio Sarsa vs Q-learning (on- vs off-policy); se deja acá por si lo querés mostrar de todos modos.

In [ ]:
puntajes_sarsa, pasos_sarsa, _ = q_learning(EP_Q, sarsa=True)
print(f"Sarsa: puntaje medio últimos {VENTANA}: {puntajes_sarsa[-VENTANA:].mean():.1f} (Q-learning: {puntajes_q[-VENTANA:].mean():.1f})")